# Solar Activity–Climate Connection: ~240-yr Cycle Analysis
# 태양활동-기후 연결: ~240년 주기 분석

**Paper / 논문**: Yang, H.-J. et al. (2019)
*J. Atmos. Sol.-Terr. Phys.*, 186, 139–146.

## 목표 / Objectives

1. **그림 2 재현** — 한/중 결합 흑점 Power Spectrum (~240yr, ~60yr, ~11yr)
2. **그림 6 재현** — 흑점 + 서리 기록 결합, ~240yr 정현파 오버레이
3. **시리즈 전체 요약** — SP-39~42의 검출 주기 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["figure.figsize"] = (14, 6)
rcParams["font.size"] = 11
rcParams["axes.grid"] = True
rcParams["grid.alpha"] = 0.3


def compute_power_spectrum(times, weights, periods):
    """Power spectrum for inhomogeneous point process (SP-39 method)."""
    sum_w = np.sum(weights)
    sum_w2 = np.sum(weights**2)
    shot_noise = sum_w2 / sum_w**2
    t_centered = times - np.mean(times)
    k_values = 2.0 * np.pi / periods
    power = np.zeros(len(periods))
    for i, k in enumerate(k_values):
        delta_k = np.sum(weights * np.exp(1j * k * t_centered)) / sum_w
        power[i] = np.abs(delta_k) ** 2 - shot_noise
    return power

## 1. 한/중 결합 흑점 데이터 / Combined Korean-Chinese Sunspot Data

표 1(한국)과 표 2(중국)에서 흑점 기록 연도를 입력합니다.
한국 55건 + 중국 134건 = 189건.

In [ ]:
# Korean sunspot years (from Table 1, pp. 141)
korean_sunspots = np.array([
    1151, 1151, 1152, 1160, 1160, 1171, 1171, 1183, 1183,
    1185, 1185, 1185, 1185, 1185, 1185, 1200, 1201, 1202,
    1204, 1204, 1204, 1258, 1258, 1278, 1356, 1356, 1361,
    1361, 1361, 1361, 1362, 1371, 1371, 1372, 1373, 1373,
    1375, 1375, 1381, 1382, 1382, 1382, 1383, 1385, 1387,
    1402, 1520, 1556, 1603, 1604, 1604, 1608, 1608, 1648, 1660,
]) + np.random.default_rng(42).uniform(0, 1, 55)

# Chinese sunspot years (from Table 2, pp. 142, representative subset)
chinese_sunspots = np.array([
    927, 947, 974, 1077, 1078, 1078, 1079, 1079, 1105, 1105,
    1112, 1118, 1118, 1120, 1120, 1122, 1129, 1131, 1136, 1137,
    1137, 1138, 1138, 1138, 1138, 1138, 1139, 1139, 1145, 1145,
    1151, 1151, 1160, 1160, 1171, 1171, 1183, 1185, 1185, 1185,
    1185, 1185, 1186, 1193, 1200, 1200, 1201, 1201, 1202, 1204,
    1204, 1204, 1205, 1228, 1258, 1258, 1275, 1276, 1278, 1356,
    1356, 1361, 1361, 1362, 1365, 1368, 1369, 1370, 1370, 1370,
    1370, 1370, 1370, 1370, 1371, 1371, 1371, 1371, 1372, 1372,
    1372, 1372, 1372, 1373, 1373, 1373, 1374, 1374, 1374, 1374,
    1375, 1375, 1375, 1376, 1381, 1381, 1382, 1382, 1383, 1387,
    1402, 1511, 1512, 1518, 1520, 1539, 1539, 1546, 1556, 1562,
    1564, 1566, 1567, 1569, 1590, 1597, 1603, 1616, 1616, 1617,
    1617, 1618, 1618, 1618, 1621, 1621, 1622, 1624, 1624, 1624,
    1625, 1625, 1625,
]) + np.random.default_rng(43).uniform(0, 1, 133)

# Combine with density equalization weights
n_k = len(korean_sunspots)
n_c = len(chinese_sunspots)
density_ratio = n_c / n_k  # Chinese/Korean density ratio

korean_weights = np.ones(n_k) * density_ratio  # Equalized
chinese_weights = np.ones(n_c)

all_sunspots = np.concatenate([korean_sunspots, chinese_sunspots])
all_weights = np.concatenate([korean_weights, chinese_weights])

print(f"Korean sunspots: {n_k} records (weight = {density_ratio:.2f})")
print(f"Chinese sunspots: {n_c} records (weight = 1.0)")
print(f"Combined: {len(all_sunspots)} records")
print(f"Year range: {all_sunspots.min():.0f} – {all_sunspots.max():.0f}")

## 2. 그림 2 재현: 한/중 결합 Power Spectrum / Reproduce Figure 2

~240년, ~60년, ~11년 주기를 확인합니다.
Verify ~240-yr, ~60-yr, and ~11-yr periods.

In [ ]:
periods = np.logspace(np.log10(5), np.log10(400), 600)
ps_combined = compute_power_spectrum(all_sunspots, all_weights, periods)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(periods, ps_combined, "k-", linewidth=0.8)

# Mark expected peaks
for period, label, color in [(240, "~240 yr", "red"), (60, "~60 yr", "blue"), (11, "~11 yr", "green")]:
    ax.axvline(x=period, color=color, linestyle="--", alpha=0.4)
    ax.annotate(label, xy=(period, np.max(ps_combined) * 0.85),
                fontsize=11, color=color, ha="center", fontweight="bold")

ax.set_xscale("log")
ax.set_xlabel("Period [Year]")
ax.set_ylabel("Power")
ax.set_title("Figure 2: Power Spectrum of Korean + Chinese Sunspot Records (927–1905)\n"
             "그림 2: 한국+중국 결합 흑점 기록의 Power Spectrum")
ax.set_xticks([6, 8, 10, 20, 40, 60, 80, 100, 200, 400])
ax.set_xticklabels(["6", "8", "10", "20", "40", "60", "80", "100", "200", "400"])
plt.tight_layout()
plt.show()

# Report peaks
for lo, hi, name in [(180, 300, "~240yr"), (40, 80, "~60yr"), (8, 15, "~11yr")]:
    mask = (periods >= lo) & (periods <= hi)
    peak = periods[mask][np.argmax(ps_combined[mask])]
    print(f"{name} peak: {peak:.1f} yr")

## 3. 그림 6 재현: 흑점 + 서리 + ~240yr 정현파 / Reproduce Figure 6

한국 흑점, 중국 흑점, 한국 서리 기록을 하나의 그래프에 결합하고
~240년 정현파를 오버레이합니다.

Combine Korean sunspots, Chinese sunspots, and Korean frost records
with ~240-yr sinusoidal overlay.

In [ ]:
# Korean sunspot histogram (per 50-yr bins)
bins_50 = np.arange(950, 1950, 50)
k_hist, _ = np.histogram(korean_sunspots, bins=bins_50)
c_hist, _ = np.histogram(chinese_sunspots, bins=bins_50)

# Frost records: approximate count per 50-yr bin (digitized from Figure 4/6)
# Frost-free period proxy: shorter = colder
frost_bins_50 = np.arange(950, 1950, 50)
frost_counts = np.array([
    0, 2, 3, 4, 5, 6, 8, 10, 8,  # 950-1400 (Goryeo sparse)
    6, 5, 4, 8, 12, 15, 18, 20, 15, 10, 8  # 1400-1900 (Joseon)
])
# Pad if needed
if len(frost_counts) < len(frost_bins_50) - 1:
    frost_counts = np.pad(frost_counts, (0, len(frost_bins_50) - 1 - len(frost_counts)))

bin_centers = (bins_50[:-1] + bins_50[1:]) / 2

# ~240yr sinusoidal curve
t_sin = np.linspace(900, 1950, 500)
sinusoid_240 = 12 + 8 * np.sin(2 * np.pi / 240 * (t_sin - 1100))

fig, ax = plt.subplots(figsize=(14, 6))

# Histograms
bar_width = 22
ax.bar(bin_centers - bar_width, k_hist, width=bar_width, color="blue",
       alpha=0.5, label="Korean sunspots / 한국 흑점", edgecolor="blue")
ax.bar(bin_centers, c_hist, width=bar_width, color="red",
       alpha=0.5, label="Chinese sunspots / 중국 흑점", edgecolor="red")
ax.bar(bin_centers + bar_width, frost_counts[:len(bin_centers)], width=bar_width,
       color="black", alpha=0.3, label="Korean frost records / 한국 서리 기록",
       edgecolor="black")

# ~240yr sinusoidal
ax.plot(t_sin, sinusoid_240, "k--", linewidth=2, alpha=0.6,
        label="~240-yr sinusoidal / ~240년 정현파")

# Grand Minima
for name, start, end, color in [
    ("Spörer", 1420, 1530, "orange"), ("Maunder", 1645, 1715, "red")
]:
    ax.axvspan(start, end, alpha=0.1, color=color)
    ax.text((start + end) / 2, ax.get_ylim()[1] * 0.05 if ax.get_ylim()[1] > 0 else 1,
            name, ha="center", fontsize=9, color=color, fontweight="bold")

ax.set_xlabel("Year")
ax.set_ylabel("Number")
ax.set_title("Figure 6: Combined Sunspot + Frost Records with ~240-yr Cycle\n"
             "그림 6: 흑점 + 서리 기록 결합 및 ~240년 주기")
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(900, 1950)
plt.tight_layout()
plt.show()

## 4. 시리즈 전체 요약: SP-39 → SP-42 / Complete Series Summary

20년간(1998–2019) 4편의 논문에서 검출된 주기와 발견을 종합합니다.
Summarize all periods detected and findings across 4 papers over 20 years (1998–2019).

In [ ]:
# Series summary visualization
fig, ax = plt.subplots(figsize=(14, 7))

papers = ["SP-39\nYang 1998", "SP-40\nLee 2004", "SP-41\nLee 2007", "SP-42\nYang 2019"]
x = np.arange(len(papers))

# Detected periods (bar chart)
schwabe = [10.5, 11.2, 11.2, 11]
gleissberg = [98, 88.4, 88.4, 0]  # 0 = not detected
cycle_60 = [0, 0, 0, 60]
cycle_240 = [0, 0, 0, 240]

bar_w = 0.2
ax.bar(x - 1.5*bar_w, schwabe, bar_w, label="Schwabe (~11 yr)",
       color="green", alpha=0.7, edgecolor="black")
ax.bar(x - 0.5*bar_w, gleissberg, bar_w, label="Gleissberg (~88 yr)",
       color="blue", alpha=0.7, edgecolor="black")
ax.bar(x + 0.5*bar_w, cycle_60, bar_w, label="~60 yr cycle",
       color="orange", alpha=0.7, edgecolor="black")
ax.bar(x + 1.5*bar_w, cycle_240, bar_w, label="~240 yr cycle (NEW)",
       color="red", alpha=0.7, edgecolor="black")

# Add data size annotations
data_info = [
    "K: 35 ss + 232 au",
    "K: 60 ss + 788 au",
    "K+C + Δ¹⁴C",
    "K+C: 189 ss\n+ 753 frost"
]
for i, info in enumerate(data_info):
    ax.text(i, -25, info, ha="center", fontsize=8, style="italic",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))

ax.set_xticks(x)
ax.set_xticklabels(papers, fontsize=10)
ax.set_ylabel("Detected Period (years)")
ax.set_title("Series Summary: Detected Solar Activity Periods (SP-39 → SP-42)\n"
             "시리즈 요약: 검출된 태양활동 주기 (SP-39 → SP-42)")
ax.legend(loc="upper left")
ax.set_ylim(-40, 280)
plt.tight_layout()
plt.show()

# Print complete summary
print("=" * 75)
print("COMPLETE SERIES SUMMARY / 시리즈 전체 요약")
print("=" * 75)
print()
print(f"{'Paper':<15} {'Year':<6} {'Data':<25} {'Key Finding':<30}")
print("-" * 75)
print(f"{'SP-39':<15} {'1998':<6} {'K: 35ss + 232au':<25} {'10.5yr + 98yr detected':<30}")
print(f"{'SP-40':<15} {'2004':<6} {'K: 60ss + 788au':<25} {'11.2yr + 88.4yr + GM':<30}")
print(f"{'SP-41':<15} {'2007':<6} {'K+C + Δ¹⁴C':<25} {'Triple validation + Spörer':<30}")
print(f"{'SP-42':<15} {'2019':<6} {'K+C: 189ss + frost':<25} {'~240yr + solar→climate':<30}")
print("=" * 75)
print()
print("Research arc / 연구 아크:")
print("  Methodology → Data Expansion → Triple Validation → Climate Connection")
print("  방법론 개발  →  데이터 확장   →   3중 검증      →  기후 연결")
print()
print("Key conclusion / 핵심 결론:")
print("  Solar activity has decreased over the last ~1000 years,")
print("  with a ~240-yr periodicity that directly correlates with")
print("  climate cooling recorded in Korean frost records.")
print("  (태양활동이 ~240년 주기로 감소하며 기후 냉각을 초래)")